In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import torch.optim.lr_scheduler as lr_scheduler
import pandas as pd 

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Hệ thống sẽ chạy trên: {device}")

Hệ thống sẽ chạy trên: cuda


In [ ]:
from sklearn.model_selection import train_test_split
import re
from collections import Counter

tokenizer = DistilBertTokenizer.from_pretrained('./my_bert_model')
# tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-multilingual-cased')

df_train = pd.read_csv('../data/train_data.csv')
df_test = pd.read_csv('../data/test_data.csv')
# df = df_full.sample(n=20000, random_state=42)

def clean_text(text):
    if not isinstance(text, str): 
        return ""

    text = re.sub(r'<.*?>', ' ', text)
    # text = re.sub(r'http\S+', 'httpaddr', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

df_train['clean_text'] = df_train['text'].apply(clean_text)
df_test['clean_text'] = df_test['text'].apply(clean_text)
# df['label_num'] = df['label'].map({'Ham': 0, 'Spam': 1})

train_texts = df_train['clean_text'].tolist()
test_texts = df_test['clean_text'].tolist()
train_labels = df_train['label'].tolist()
test_labels = df_test['label'].tolist()

# train_texts, test_texts, train_labels, test_labels = train_test_split(
#     all_texts, all_labels, test_size=0.2, random_state=42
# )

train_encodings = tokenizer(train_texts, truncation=True, padding='max_length', max_length=256, return_tensors='pt')
test_encodings = tokenizer(test_texts, truncation=True, padding='max_length', max_length=256, return_tensors='pt')

class SpamDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SpamDataset(train_encodings, train_labels)
test_dataset = SpamDataset(test_encodings, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=True) 

In [3]:
model = DistilBertForSequenceClassification.from_pretrained('./my_bert_model', num_labels=2)
# model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-multilingual-cased', num_labels=2)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

scheduler = lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=2, T_mult=1)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: ./my_bert_model
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
from tqdm import tqdm
import time
import torch

epochs = 4  

for epoch in range(epochs):
    print(f"\n========== VÒNG {epoch + 1}/{epochs} ==========")
    start_time = time.time()
    
    model.train()
    total_loss = 0
    
    train_iterator = tqdm(train_loader, desc=f"Đang chạy Vòng {epoch+1}")
    
    for batch in train_iterator:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        
        train_iterator.set_postfix({'Loss': f"{loss.item():.4f}"})
        
    scheduler.step()
    
    model.eval()
    correct_predictions = 0
    total_test_emails = 0
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Chấm điểm"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask)
            predictions = torch.argmax(outputs.logits, dim=1)
            
            correct_predictions += (predictions == labels).sum().item()
            total_test_emails += labels.size(0)
            
    test_accuracy = (correct_predictions / total_test_emails) * 100
    avg_train_loss = total_loss / len(train_loader)
    end_time = time.time()
    
    print(f"\nKẾT QUẢ VÒNG {epoch + 1} | Mất {end_time - start_time:.1f} giây")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"ĐỘ CHÍNH XÁC: {test_accuracy:.2f}%")


========== VÒNG 1/4 ==========


Chấm điểm: 100%|██████████| 857/857 [00:54<00:00, 15.80it/s]



KẾT QUẢ VÒNG 1 | Mất 898.2 giây
Train Loss: 0.0673
ĐỘ CHÍNH XÁC: 99.11%

========== VÒNG 2/4 ==========


Chấm điểm: 100%|██████████| 857/857 [00:54<00:00, 15.73it/s]



KẾT QUẢ VÒNG 2 | Mất 896.7 giây
Train Loss: 0.0111
ĐỘ CHÍNH XÁC: 98.83%

========== VÒNG 3/4 ==========


Chấm điểm: 100%|██████████| 857/857 [00:53<00:00, 15.90it/s]



KẾT QUẢ VÒNG 3 | Mất 895.9 giây
Train Loss: 0.0097
ĐỘ CHÍNH XÁC: 99.23%

========== VÒNG 4/4 ==========


Chấm điểm: 100%|██████████| 857/857 [00:54<00:00, 15.75it/s]


KẾT QUẢ VÒNG 4 | Mất 896.3 giây
Train Loss: 0.0031
ĐỘ CHÍNH XÁC: 99.22%


In [7]:
import torch
torch.save(model.state_dict(), 'TEST_spam_Transformer_model.pth')
print("Đã lưu vào file 'TEST_spam_Transformer_model.pth'!")

Đã lưu vào file 'TEST_spam_Transformer_model.pth'!
